In [1]:
import os
import asyncio
import httpx

from dotenv import find_dotenv, load_dotenv

In [2]:
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete, openai_embed

2025-05-10 10:43:12 - pipmaster.package_manager - INFO - Targeting pip associated with Python: /workspaces/devcontainers/llm-101/llm_101/rag/light-rag-agent/LightRAG/.venv/bin/python | Command base: /workspaces/devcontainers/llm-101/llm_101/rag/light-rag-agent/LightRAG/.venv/bin/python -m pip


In [3]:
from langchain.text_splitter import CharacterTextSplitter

In [4]:
# required to have async functions work in Jupyter
import nest_asyncio
nest_asyncio.apply()

In [5]:
# Load environment variables from .env file
found_env_file = load_dotenv(find_dotenv())
if not found_env_file:
    print("WARNING: No .env file found. Please create one with your OpenAI API key.")

WORKING_DIR = "./pydantic-docs"
PYDANTIC_DOCS_URL = "https://ai.pydantic.dev/llms.txt"
PYDANTIC_DOCS_PATH = os.path.join(WORKING_DIR, "pydantic_docs.txt")

## Retrieve pydantic docs

In [6]:
def fetch_pydantic_docs() -> str:
    """Fetch the Pydantic AI documentation from the URL.
    
    Returns:
        The content of the documentation
    """
    try:
        response = httpx.get(PYDANTIC_DOCS_URL)
        response.raise_for_status()
        return response.text
    except Exception as e:
        raise Exception(f"Error fetching Pydantic AI documentation: {e}")

In [7]:
os.makedirs(WORKING_DIR, exist_ok=True)
files = os.listdir(WORKING_DIR)

if os.path.exists(PYDANTIC_DOCS_PATH):
    print("Pydantic docs already exist.")
else:
    print("Pydantic docs do not exist.")
    docs = fetch_pydantic_docs()
    with open(os.path.join(WORKING_DIR, "pydantic_docs.txt"), "w") as f:
        f.write(docs)
    print(f"Pydantic docs fetched and saved to {PYDANTIC_DOCS_PATH}")

Pydantic docs already exist.


## Insert Into KG

### Text Chunking

In [8]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

with open(PYDANTIC_DOCS_PATH, "r") as f:
    docs_str = f.read()

documents = text_splitter.split_text(docs_str)
print(f"Number of documents: {len(documents)}")
[print(len(documents[x])) for x in range(len(documents))]

Number of documents: 3
642
887
992


[None, None, None]

### Initialize LightRAG

In [9]:
rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=openai_complete,
    llm_model_name="gpt-4o-mini",
    embedding_func=openai_embed,
    embedding_cache_config={
        "enabled": False,
        "similarity_threshold": 0.95,
        "use_llm_check": False,
    },
    chunk_token_size=1000,
    chunk_overlap_token_size=50,
)


INFO: Process 77638 Shared-Data created for Single Process
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': './pydantic-docs/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': './pydantic-docs/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': './pydantic-docs/vdb_chunks.json'} 0 data


INFO: Process 77638 initialized updated flags for namespace: [full_docs]
INFO: Process 77638 ready to initialize storage namespace: [full_docs]
INFO: Process 77638 initialized updated flags for namespace: [text_chunks]
INFO: Process 77638 ready to initialize storage namespace: [text_chunks]
INFO: Process 77638 initialized updated flags for namespace: [entities]
INFO: Process 77638 initialized updated flags for namespace: [relationships]
INFO: Process 77638 initialized updated flags for namespace: [chunks]
INFO: Process 77638 initialized updated flags for namespace: [chunk_entity_relation]
INFO: Process 77638 initialized updated flags for namespace: [llm_response_cache]
INFO: Process 77638 ready to initialize storage namespace: [llm_response_cache]
INFO: Process 77638 initialized updated flags for namespace: [doc_status]
INFO: Process 77638 ready to initialize storage namespace: [doc_status]


In [10]:
asyncio.run(rag.initialize_storages())
await rag.initialize_storages()
rag._storages_status

<StoragesStatus.INITIALIZED: 'initialized'>

In [11]:
from lightrag.kg.shared_storage import initialize_pipeline_status

In [12]:
await initialize_pipeline_status()

INFO: Process 77638 Pipeline namespace initialized


In [13]:
rag.insert(documents, file_paths=[PYDANTIC_DOCS_PATH for _ in range(len(documents))])

In [15]:
rag._storages_status

<StoragesStatus.INITIALIZED: 'initialized'>

In [21]:
response = await rag.aquery("What is the purpose of Pydantic?", param=QueryParam(mode="local", top_k=3))
print(response)

### Purpose of Pydantic

Pydantic is primarily a data validation and settings management tool used in Python programming. It is especially beneficial when working with frameworks like FastAPI. Its key functionalities include:

- **Data Validation**: Pydantic ensures that the data provided to your application conforms to the expected types and formats, which helps in maintaining data integrity.
  
- **Settings Management**: It offers an effective way to manage application settings, making it easier to configure and validate environment variables and application configuration settings.

By using Pydantic, developers can create robust applications with a focus on correctness and efficiency, especially when dealing with complex data structures and ensuring that inputs are validated according to specified rules.

### References
1. [KG] ./pydantic-docs/pydantic_docs.txt
2. [KG] ./pydantic-docs/pydantic_docs.txt
3. [KG] ./pydantic-docs/pydantic_docs.txt


In [16]:
rag.insert("Evidence that this is a test")

In [33]:
rag._storages_status

<StoragesStatus.INITIALIZED: 'initialized'>

In [17]:
await rag.finalize_storages()
rag._storages_status

<StoragesStatus.FINALIZED: 'finalized'>

In [19]:
print(rag.query("Is there any evidence in the knowledge graph that this is a test?"))

Yes, there is evidence in the knowledge base indicating that this is a test. Specifically, a relationship labeled "test" correlates with the mention of its relevance as a central theme in the context.

### Key Evidence
- The relationship notes the repeat mention of "test" as significant, implying it is a notable aspect within the provided information.

### Conclusion
This points towards the context being recognized as a test scenario or related to testing practices.

### References
- [KG] unknown_source
